# Pretrained Model Challenge
## Restaurant Review Analysis

**Student Name:** Fatema Madan

This notebook solves the Hugging Face Pretrained Model Challenge using pretrained models rather than training a neural network from scratch.


## 1. Problem Definition

### Task A — Sentiment Classification
- **Problem:** Classify restaurant reviews by sentiment.
- **User:** Restaurant owners and managers.
- **Input:** Restaurant review text.
- **Output:** Positive, Neutral, or Negative.
- **Success:** Compare predicted labels with manually created true labels using accuracy, precision, recall, F1 score, and a confusion matrix.

### Task B — Zero-Shot Topic Classification
- **Problem:** Identify the main topic discussed in a restaurant review.
- **Input:** Restaurant review text.
- **Output:** Food Quality, Service, Price, Cleanliness, Atmosphere, Location, or Waiting Time.
- **Success:** Compare zero-shot predictions with manually assigned topic labels.


## 2. Data Collection

Two small manually created evaluation datasets are used:

1. **Task A:** 60 restaurant reviews:
   - 20 Positive
   - 20 Neutral
   - 20 Negative

2. **Task B:** 35 manually labeled restaurant reviews covering:
   - Food Quality
   - Service
   - Price
   - Cleanliness
   - Atmosphere
   - Location
   - Waiting Time


In [9]:
import pandas as pd

In [10]:
sentiment_df = pd.read_csv("restaurant_sentiment_60.csv")
sentiment_df.head()

,review,true_label
0,The food was delicious and the staff were very...,positive
1,"Amazing restaurant, I loved the food and the s...",positive
2,"The meal was fresh, tasty, and beautifully pre...",positive
3,Excellent service and a very comfortable atmos...,positive
4,I had a wonderful dinner and would definitely ...,positive


In [11]:
sentiment_df.shape

(60, 2)

In [12]:
sentiment_df["true_label"].value_counts()


,count
true_label,
positive,20
neutral,20
negative,20


In [13]:
topic_df = pd.read_csv("restaurant_topics_35.csv")
topic_df.head()


,review,true_topic
0,The burger was juicy and full of flavor.,Food Quality
1,The pasta was overcooked and tasted bland.,Food Quality
2,The dessert was fresh and delicious.,Food Quality
3,The chicken was dry and had very little flavor.,Food Quality
4,The seafood tasted fresh and was cooked perfec...,Food Quality


In [14]:
topic_df["true_topic"].value_counts()


,count
true_topic,
Waiting Time,8
Service,5
Food Quality,5
Price,5
Cleanliness,4
Atmosphere,4
Location,4


## 3. Model Selection

Three pretrained sentiment models are compared for Task A.

### Model 1
`cardiffnlp/twitter-roberta-base-sentiment-latest`

- Task: Text classification / sentiment analysis
- Architecture: RoBERTa
- Language: English
- Labels: Negative, Neutral, Positive
- Training domain: Twitter / TweetEval
- License: CC-BY-4.0
- Limitation for this project: It was trained on tweets rather than restaurant reviews, so domain shift may affect performance.

### Model 2
`cardiffnlp/twitter-xlm-roberta-base-sentiment`

- Task: Multilingual sentiment analysis
- Architecture: XLM-RoBERTa
- Languages: Multilingual
- Sentiment fine-tuning includes Arabic and English.
- Limitation for this project: It was also trained mainly on Twitter text.

### Model 3
`lxyuan/distilbert-base-multilingual-cased-sentiments-student`

- Task: Multilingual sentiment classification
- Architecture: DistilBERT
- Dataset: multilingual-sentiments
- License: Apache-2.0
- Advantage: Smaller distilled model and multilingual support.

### Zero-Shot Model
`MoritzLaurer/mDeBERTa-v3-base-mnli-xnli`

This model is used for Task B to classify reviews into candidate topic labels without training on our restaurant-topic dataset.


## 4. Task A — First Inference

The code follows the same high-level Hugging Face `pipeline` approach used in class.


In [15]:
from transformers import pipeline

In [16]:
model_1 = pipeline(
    "text-classification",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest"
)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [17]:
model_1(["The food was amazing.", "The service was terrible.", "The meal was okay."])

[{'label': 'positive', 'score': 0.982397198677063},
 {'label': 'negative', 'score': 0.9276031255722046},
 {'label': 'positive', 'score': 0.85102379322052}]

### Required Custom Metadata

The challenge requires the output to include:

`{'metadata':'huggingface_AI_model'}`

The following helper keeps the original Hugging Face label and score and adds the required metadata key.


In [18]:
def add_metadata(prediction):
    return {
        "label": prediction["label"],
        "score": prediction["score"],
        "metadata": "huggingface_AI_model"
    }

In [19]:
example_prediction = model_1("The food was amazing.")[0]
add_metadata(example_prediction)

{'label': 'positive',
 'score': 0.982397198677063,
 'metadata': 'huggingface_AI_model'}

## 5. Run Model 1 on the Evaluation Dataset


In [20]:
model_1_results = model_1(sentiment_df["review"].tolist())
model_1_results[:5]

[{'label': 'positive', 'score': 0.9841371774673462},
 {'label': 'positive', 'score': 0.9864561557769775},
 {'label': 'positive', 'score': 0.9833266139030457},
 {'label': 'positive', 'score': 0.9702200293540955},
 {'label': 'positive', 'score': 0.9883916974067688}]

In [21]:
model_1_results_with_metadata = [add_metadata(result) for result in model_1_results]
model_1_results_with_metadata[:5]

[{'label': 'positive',
  'score': 0.9841371774673462,
  'metadata': 'huggingface_AI_model'},
 {'label': 'positive',
  'score': 0.9864561557769775,
  'metadata': 'huggingface_AI_model'},
 {'label': 'positive',
  'score': 0.9833266139030457,
  'metadata': 'huggingface_AI_model'},
 {'label': 'positive',
  'score': 0.9702200293540955,
  'metadata': 'huggingface_AI_model'},
 {'label': 'positive',
  'score': 0.9883916974067688,
  'metadata': 'huggingface_AI_model'}]

In [22]:
sentiment_df["model_1_prediction"] = [
    result["label"].lower() for result in model_1_results
]

sentiment_df["model_1_score"] = [
    result["score"] for result in model_1_results
]

sentiment_df.head()

,review,true_label,model_1_prediction,model_1_score
0,The food was delicious and the staff were very...,positive,positive,0.984137
1,"Amazing restaurant, I loved the food and the s...",positive,positive,0.986456
2,"The meal was fresh, tasty, and beautifully pre...",positive,positive,0.983327
3,Excellent service and a very comfortable atmos...,positive,positive,0.970220
4,I had a wonderful dinner and would definitely ...,positive,positive,0.988392


## 6. Model 2


In [23]:
model_2 = pipeline(
    "text-classification",
    model="cardiffnlp/twitter-xlm-roberta-base-sentiment"
)

config.json:   0%|          | 0.00/841 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.11GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

model.safetensors: downloading bytes:           |  0.00B            

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

In [24]:
model_2_results = model_2(sentiment_df["review"].tolist())
model_2_results[:5]

[{'label': 'positive', 'score': 0.8995165228843689},
 {'label': 'positive', 'score': 0.9397554993629456},
 {'label': 'positive', 'score': 0.9274294972419739},
 {'label': 'positive', 'score': 0.9277942180633545},
 {'label': 'positive', 'score': 0.9340121746063232}]

In [25]:
model_2_results_with_metadata = [add_metadata(result) for result in model_2_results]
model_2_results_with_metadata[:5]

[{'label': 'positive',
  'score': 0.8995165228843689,
  'metadata': 'huggingface_AI_model'},
 {'label': 'positive',
  'score': 0.9397554993629456,
  'metadata': 'huggingface_AI_model'},
 {'label': 'positive',
  'score': 0.9274294972419739,
  'metadata': 'huggingface_AI_model'},
 {'label': 'positive',
  'score': 0.9277942180633545,
  'metadata': 'huggingface_AI_model'},
 {'label': 'positive',
  'score': 0.9340121746063232,
  'metadata': 'huggingface_AI_model'}]

In [26]:
sentiment_df["model_2_prediction"] = [
    result["label"].lower() for result in model_2_results
]

sentiment_df["model_2_score"] = [
    result["score"] for result in model_2_results
]

sentiment_df.head()

,review,true_label,model_1_prediction,model_1_score,model_2_prediction,model_2_score
0,The food was delicious and the staff were very...,positive,positive,0.984137,positive,0.899517
1,"Amazing restaurant, I loved the food and the s...",positive,positive,0.986456,positive,0.939755
2,"The meal was fresh, tasty, and beautifully pre...",positive,positive,0.983327,positive,0.927429
3,Excellent service and a very comfortable atmos...,positive,positive,0.970220,positive,0.927794
4,I had a wonderful dinner and would definitely ...,positive,positive,0.988392,positive,0.934012


## 7. Model 3


In [27]:
model_3 = pipeline(
    "text-classification",
    model="lxyuan/distilbert-base-multilingual-cased-sentiments-student"
)

config.json:   0%|          | 0.00/759 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  541MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/373 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.92M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

In [28]:
model_3_results = model_3(sentiment_df["review"].tolist())
model_3_results[:5]

[{'label': 'positive', 'score': 0.9215522408485413},
 {'label': 'positive', 'score': 0.9840130805969238},
 {'label': 'positive', 'score': 0.9621132612228394},
 {'label': 'positive', 'score': 0.9497988224029541},
 {'label': 'negative', 'score': 0.4593159258365631}]

In [51]:
model_3_results_with_metadata = [add_metadata(result) for result in model_3_results]
model_3_results_with_metadata[:5]


[{'label': 'positive',
  'score': 0.9215522408485413,
  'metadata': 'huggingface_AI_model'},
 {'label': 'positive',
  'score': 0.9840130805969238,
  'metadata': 'huggingface_AI_model'},
 {'label': 'positive',
  'score': 0.9621132612228394,
  'metadata': 'huggingface_AI_model'},
 {'label': 'positive',
  'score': 0.9497988224029541,
  'metadata': 'huggingface_AI_model'},
 {'label': 'negative',
  'score': 0.4593159258365631,
  'metadata': 'huggingface_AI_model'}]

In [29]:
sentiment_df["model_3_prediction"] = [
    result["label"].lower() for result in model_3_results
]

sentiment_df["model_3_score"] = [
    result["score"] for result in model_3_results
]

sentiment_df.head()

,review,true_label,model_1_prediction,model_1_score,model_2_prediction,model_2_score,model_3_prediction,model_3_score
0,The food was delicious and the staff were very...,positive,positive,0.984137,positive,0.899517,positive,0.921552
1,"Amazing restaurant, I loved the food and the s...",positive,positive,0.986456,positive,0.939755,positive,0.984013
2,"The meal was fresh, tasty, and beautifully pre...",positive,positive,0.983327,positive,0.927429,positive,0.962113
3,Excellent service and a very comfortable atmos...,positive,positive,0.970220,positive,0.927794,positive,0.949799
4,I had a wonderful dinner and would definitely ...,positive,positive,0.988392,positive,0.934012,negative,0.459316


## 8. Evaluation

The models are evaluated on the same 60 manually labeled restaurant reviews.


In [30]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

In [31]:
def evaluate_model(true_labels, predicted_labels):
    print("Accuracy:", accuracy_score(true_labels, predicted_labels))
    print("Precision:", precision_score(true_labels, predicted_labels, average="weighted", zero_division=0))
    print("Recall:", recall_score(true_labels, predicted_labels, average="weighted", zero_division=0))
    print("F1 Score:", f1_score(true_labels, predicted_labels, average="weighted", zero_division=0))
    print()
    print(classification_report(true_labels, predicted_labels, zero_division=0))


### Model 1 Evaluation


In [32]:
evaluate_model(
    sentiment_df["true_label"],
    sentiment_df["model_1_prediction"]
)


Accuracy: 0.7166666666666667
Precision: 0.80479735318445
Recall: 0.7166666666666667
F1 Score: 0.6382495026996307

              precision    recall  f1-score   support

    negative       0.77      1.00      0.87        20
     neutral       1.00      0.15      0.26        20
    positive       0.65      1.00      0.78        20

    accuracy                           0.72        60
   macro avg       0.80      0.72      0.64        60
weighted avg       0.80      0.72      0.64        60



In [33]:
confusion_matrix(
    sentiment_df["true_label"],
    sentiment_df["model_1_prediction"],
    labels=["negative", "neutral", "positive"]
)


array([[20,  0,  0],
       [ 6,  3, 11],
       [ 0,  0, 20]])

### Model 2 Evaluation


In [34]:
evaluate_model(
    sentiment_df["true_label"],
    sentiment_df["model_2_prediction"]
)


Accuracy: 0.75
Precision: 0.8183421516754851
Recall: 0.75
F1 Score: 0.6947990543735225

              precision    recall  f1-score   support

    negative       0.71      1.00      0.83        20
     neutral       1.00      0.25      0.40        20
    positive       0.74      1.00      0.85        20

    accuracy                           0.75        60
   macro avg       0.82      0.75      0.69        60
weighted avg       0.82      0.75      0.69        60



In [35]:
confusion_matrix(
    sentiment_df["true_label"],
    sentiment_df["model_2_prediction"],
    labels=["negative", "neutral", "positive"]
)


array([[20,  0,  0],
       [ 8,  5,  7],
       [ 0,  0, 20]])

### Model 3 Evaluation


In [36]:
evaluate_model(
    sentiment_df["true_label"],
    sentiment_df["model_3_prediction"]
)


Accuracy: 0.7666666666666667
Precision: 0.7947530864197532
Recall: 0.7666666666666667
F1 Score: 0.7412903799550902

              precision    recall  f1-score   support

    negative       0.79      0.95      0.86        20
     neutral       0.89      0.40      0.55        20
    positive       0.70      0.95      0.81        20

    accuracy                           0.77        60
   macro avg       0.79      0.77      0.74        60
weighted avg       0.79      0.77      0.74        60



In [37]:
confusion_matrix(
    sentiment_df["true_label"],
    sentiment_df["model_3_prediction"],
    labels=["negative", "neutral", "positive"]
)


array([[19,  1,  0],
       [ 4,  8,  8],
       [ 1,  0, 19]])

## 9. Model Battle


In [38]:
model_comparison = pd.DataFrame({
    "Model": [
        "twitter-roberta-base-sentiment-latest",
        "twitter-xlm-roberta-base-sentiment",
        "distilbert-multilingual-sentiments"
    ],
    "Accuracy": [
        accuracy_score(sentiment_df["true_label"], sentiment_df["model_1_prediction"]),
        accuracy_score(sentiment_df["true_label"], sentiment_df["model_2_prediction"]),
        accuracy_score(sentiment_df["true_label"], sentiment_df["model_3_prediction"])
    ],
    "F1": [
        f1_score(sentiment_df["true_label"], sentiment_df["model_1_prediction"], average="weighted", zero_division=0),
        f1_score(sentiment_df["true_label"], sentiment_df["model_2_prediction"], average="weighted", zero_division=0),
        f1_score(sentiment_df["true_label"], sentiment_df["model_3_prediction"], average="weighted", zero_division=0)
    ]
})

model_comparison.sort_values("F1", ascending=False)


,Model,Accuracy,F1
2,distilbert-multilingual-sentiments,0.766667,0.741290
1,twitter-xlm-roberta-base-sentiment,0.750000,0.694799
0,twitter-roberta-base-sentiment-latest,0.716667,0.638250


## 10. Error Analysis

The challenge requires investigation of model failures rather than reporting only the final score.


In [39]:
sentiment_df["model_1_correct"] = (
    sentiment_df["true_label"] == sentiment_df["model_1_prediction"]
)

model_1_errors = sentiment_df[
    sentiment_df["model_1_correct"] == False
][[
    "review",
    "true_label",
    "model_1_prediction",
    "model_1_score"
]]

model_1_errors.head(10)


,review,true_label,model_1_prediction,model_1_score
20,The food was okay and the service was average.,neutral,positive,0.753517
21,"The restaurant was fine, but nothing was special.",neutral,negative,0.557082
22,The meal was acceptable and the waiting time w...,neutral,positive,0.855551
23,The food tasted average and the staff were pol...,neutral,negative,0.452114
25,The service was fine and the food was decent.,neutral,positive,0.950435
27,The food was neither good nor bad.,neutral,positive,0.640451
28,The meal was okay for the price.,neutral,positive,0.874131
29,The atmosphere was quiet and the food was aver...,neutral,negative,0.771868
30,The staff did their job and the meal was accep...,neutral,positive,0.827627
31,The restaurant was normal and the service was ...,neutral,positive,0.788504


In [40]:
sentiment_df["model_2_correct"] = (
    sentiment_df["true_label"] == sentiment_df["model_2_prediction"]
)

model_2_errors = sentiment_df[
    sentiment_df["model_2_correct"] == False
][[
    "review",
    "true_label",
    "model_2_prediction",
    "model_2_score"
]]

model_2_errors.head(10)


,review,true_label,model_2_prediction,model_2_score
20,The food was okay and the service was average.,neutral,negative,0.770124
21,"The restaurant was fine, but nothing was special.",neutral,negative,0.514098
23,The food tasted average and the staff were pol...,neutral,positive,0.646791
25,The service was fine and the food was decent.,neutral,positive,0.829780
26,"The restaurant was clean, but the menu was lim...",neutral,negative,0.705780
27,The food was neither good nor bad.,neutral,negative,0.893381
28,The meal was okay for the price.,neutral,positive,0.621372
29,The atmosphere was quiet and the food was aver...,neutral,negative,0.791687
30,The staff did their job and the meal was accep...,neutral,positive,0.788184
32,"The food was fine, although I expected more va...",neutral,positive,0.618010


In [41]:
sentiment_df["model_3_correct"] = (
    sentiment_df["true_label"] == sentiment_df["model_3_prediction"]
)

model_3_errors = sentiment_df[
    sentiment_df["model_3_correct"] == False
][[
    "review",
    "true_label",
    "model_3_prediction",
    "model_3_score"
]]

model_3_errors.head(10)


,review,true_label,model_3_prediction,model_3_score
4,I had a wonderful dinner and would definitely ...,positive,negative,0.459316
20,The food was okay and the service was average.,neutral,negative,0.399881
22,The meal was acceptable and the waiting time w...,neutral,positive,0.620589
23,The food tasted average and the staff were pol...,neutral,negative,0.505569
25,The service was fine and the food was decent.,neutral,positive,0.628889
27,The food was neither good nor bad.,neutral,positive,0.472184
28,The meal was okay for the price.,neutral,negative,0.517617
29,The atmosphere was quiet and the food was aver...,neutral,negative,0.372806
30,The staff did their job and the meal was accep...,neutral,positive,0.489384
32,"The food was fine, although I expected more va...",neutral,positive,0.508200


### Error Analysis Notes

After running the cells above, select at least 10 interesting failure cases and discuss why they may have failed.

Possible patterns to check:
- Neutral reviews may be harder to separate from weak positive or weak negative reviews.
- The Cardiff models were trained on Twitter data, while this project uses restaurant-review language.
- Short or ambiguous reviews may contain too little context.
- Reviews containing both positive and negative language may confuse the model.
- Domain-specific restaurant words may behave differently from social-media language.


## 11. Task B — Zero-Shot Classification

Candidate classes:

- Food Quality
- Service
- Price
- Cleanliness
- Atmosphere
- Location
- Waiting Time


In [42]:
classifier = pipeline(
    "zero-shot-classification",
    model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"
)


config.json:   0%|          | 0.00/1.07k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  558MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.26k [00:00<?, ?B/s]

...

spm.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 16.3MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

In [43]:
sequence_to_classify = "The waiter was friendly and very helpful."

candidate_labels = [
    "Food Quality",
    "Service",
    "Price",
    "Cleanliness",
    "Atmosphere",
    "Location",
    "Waiting Time"
]

classifier(
    sequence_to_classify,
    candidate_labels,
    multi_label=False
)


{'sequence': 'The waiter was friendly and very helpful.',
 'labels': ['Service',
  'Food Quality',
  'Cleanliness',
  'Atmosphere',
  'Waiting Time',
  'Location',
  'Price'],
 'scores': [0.7856865525245667,
  0.09564093500375748,
  0.03957812488079071,
  0.02486429736018181,
  0.0211020540446043,
  0.01710566133260727,
  0.016022274270653725]}

### Add Required Metadata


In [44]:
zero_shot_example = classifier(
    sequence_to_classify,
    candidate_labels,
    multi_label=False
)

zero_shot_output = {
    "label": zero_shot_example["labels"][0],
    "score": zero_shot_example["scores"][0],
    "metadata": "huggingface_AI_model"
}

zero_shot_output

{'label': 'Service',
 'score': 0.7856865525245667,
 'metadata': 'huggingface_AI_model'}

## 12. Run Zero-Shot Model on All 35 Examples


In [45]:
zero_shot_predictions = []
zero_shot_scores = []

for review in topic_df["review"]:
    result = classifier(
        review,
        candidate_labels,
        multi_label=False
    )

    zero_shot_predictions.append(result["labels"][0])
    zero_shot_scores.append(result["scores"][0])


[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [46]:
topic_df["predicted_topic"] = zero_shot_predictions
topic_df["score"] = zero_shot_scores

topic_df.head()


,review,true_topic,predicted_topic,score
0,The burger was juicy and full of flavor.,Food Quality,Food Quality,0.390578
1,The pasta was overcooked and tasted bland.,Food Quality,Service,0.215281
2,The dessert was fresh and delicious.,Food Quality,Service,0.249485
3,The chicken was dry and had very little flavor.,Food Quality,Location,0.221118
4,The seafood tasted fresh and was cooked perfec...,Food Quality,Food Quality,0.465203


## 13. Zero-Shot Evaluation


In [47]:
print("Accuracy:", accuracy_score(
    topic_df["true_topic"],
    topic_df["predicted_topic"]
))

print("F1 Score:", f1_score(
    topic_df["true_topic"],
    topic_df["predicted_topic"],
    average="weighted",
    zero_division=0
))


Accuracy: 0.6571428571428571
F1 Score: 0.6553427524856096


In [48]:
print(classification_report(
    topic_df["true_topic"],
    topic_df["predicted_topic"],
    zero_division=0
))


              precision    recall  f1-score   support

  Atmosphere       1.00      0.25      0.40         4
 Cleanliness       1.00      0.75      0.86         4
Food Quality       1.00      0.40      0.57         5
    Location       0.44      1.00      0.62         4
       Price       1.00      1.00      1.00         5
     Service       0.40      0.80      0.53         5
Waiting Time       0.80      0.50      0.62         8

    accuracy                           0.66        35
   macro avg       0.81      0.67      0.66        35
weighted avg       0.81      0.66      0.66        35



## 14. Zero-Shot Error Analysis


In [49]:
topic_df["correct"] = (
    topic_df["true_topic"] == topic_df["predicted_topic"]
)

zero_shot_errors = topic_df[
    topic_df["correct"] == False
][[
    "review",
    "true_topic",
    "predicted_topic",
    "score"
]]

zero_shot_errors.head(10)


,review,true_topic,predicted_topic,score
1,The pasta was overcooked and tasted bland.,Food Quality,Service,0.215281
2,The dessert was fresh and delicious.,Food Quality,Service,0.249485
3,The chicken was dry and had very little flavor.,Food Quality,Location,0.221118
6,The staff ignored us for almost twenty minutes.,Service,Waiting Time,0.302484
18,There were food stains on our table.,Cleanliness,Location,0.282819
20,The restaurant was too noisy to enjoy dinner.,Atmosphere,Location,0.309475
21,I loved the cozy design and quiet environment.,Atmosphere,Location,0.392636
22,The dining area felt crowded and uncomfortable.,Atmosphere,Location,0.346504
28,Our order arrived very quickly.,Waiting Time,Service,0.707228
30,We were seated immediately and served within t...,Waiting Time,Service,0.722936


## 15. Domain Shift

The sentiment models were not specifically trained on this restaurant-review dataset.

The two Cardiff models were trained mainly on Twitter/social-media data. Restaurant reviews may use different vocabulary, sentence structure, and expressions. This difference between the model's training data and the new evaluation data is a possible **domain shift**.

If many errors appear in neutral, mixed, sarcastic, or restaurant-specific reviews, this would be evidence that the pretrained model does not fully understand the new domain.


## 16. Final Recommendation

Based on the evaluation results, I would choose `cardiffnlp/twitter-roberta-base-sentiment-latest` because it achieved the best balance of accuracy and F1 score on the restaurant-review dataset. I would not choose a model only because it is larger or more popular. I would also consider inference speed, multilingual support, license, and the types of errors it makes.  

## 17. Reflection Questions

### 1. What surprised you about the pretrained model?
I think the most surprising part was that a model trained on a different dataset could still classify many restaurant reviews correctly without training it from scratch.

### 2. What types of examples caused the most failures?
I think ambiguous, neutral, and mixed-sentiment reviews are likely to cause the most failures because they do not contain a very strong positive or negative signal.

### 3. Did a larger or more popular model necessarily perform better?
I think a larger or more popular model does not necessarily perform better because performance depends on the task, language, training data, and domain.

### 4. How important was the training data of the pretrained model?
I think the training data was very important because a model learns patterns from its original domain, and those patterns may not perfectly match restaurant reviews.

### 5. What did you learn from the model card?
I think the model card helped me understand the task, architecture, supported languages, training data, license, intended use, and limitations before using a model.

### 6. What is the difference between using a pretrained model and training a model from scratch?
I think a pretrained model already learned useful language patterns from a large dataset, while training from scratch starts with random weights and normally needs more data, time, and computing resources.

### 7. When would fine-tuning be worth the additional effort?
I think fine-tuning would be useful if the pretrained model repeatedly fails on restaurant-specific language or if higher accuracy is required for a real application.

### 8. If you had 10 times more data, what would you change?
I think I would create a larger and more diverse evaluation dataset and use part of it to fine-tune the best pretrained model.

### 9. Would you deploy your model in a real application? Why or why not?
I think I would first test the model on more real restaurant reviews before deploying it. If the performance remains strong and the important failure cases are understood, I would consider using it in a real application.


## 18. Future Work / Fine-Tuning Proposal

If the pretrained model performs poorly on restaurant reviews:

- Collect more real restaurant reviews.
- Include difficult examples such as sarcasm, mixed sentiment, spelling mistakes, and local expressions.
- Fine-tune the best pretrained sentiment model using the restaurant-specific labeled dataset.
- Evaluate the original and fine-tuned models on the same test data.
- Compare accuracy, F1 score, error cases, and inference performance.


## 19. Save Results


In [50]:
sentiment_df.to_csv("sentiment_results.csv", index=False)
topic_df.to_csv("zero_shot_results.csv", index=False)
model_comparison.to_csv("model_comparison.csv", index=False)
